# GPT-OSS 120B GRPO Text-to-SQL on RunPod

This notebook sets up reinforcement learning fine-tuning of **unsloth/gpt-oss-120b** for the BIRD text-to-SQL benchmark. It is structured so you can prepare everything in a CPU-only environment and then execute training on a RunPod instance with **4× A100 80GB** GPUs. The heavy training cell is left for you to run once you copy the notebook to RunPod.



> **TL;DR**
> 1. Use this notebook to prepare data, reward functions, and configuration.
> 2. Launch a RunPod instance with 4×A100 80GB, copy the notebook, and run the training cell there.
> 3. Set `BIRD_DATABASE_ROOT` to the location of the extracted BIRD databases so execution-based rewards can run.



### Installation

The cell below installs Unsloth with the optional torch build plus text-to-SQL utilities. Skip it on RunPod if your environment already has matching versions.



In [ ]:
%%capture
!pip install -q "unsloth[torch]>=2024.8.0" datasets sqlglot duckdb sqlite-utils accelerate bitsandbytes transformers huggingface_hub



### Environment checks

Confirm that the notebook sees every GPU before loading GPT-OSS 120B. On RunPod you should see four A100 80GB devices.



In [ ]:
import os, torch, subprocess

def describe_gpus():
    if torch.cuda.is_available():
        print(f"Visible CUDA devices: {torch.cuda.device_count()}")
        for idx in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(idx)
            print(f"GPU {idx}: {props.name} — {props.total_memory/1024**3:.1f} GB")
        try:
            subprocess.run(["nvidia-smi"], check=False)
        except FileNotFoundError:
            print("nvidia-smi not found on this host.")
    else:
        print("CUDA is not available. Model loading will fall back to CPU offload which is not practical for training GPT-OSS 120B.")

describe_gpus()



### Load GPT-OSS 120B with memory-aware configuration

We quantize weights to 4-bit and split them across the available GPUs. For CPU-only execution we enable offloading, but full training requires 4×A100 80GB.



In [ ]:
from unsloth import FastLanguageModel
import torch
from pathlib import Path

max_seq_length = 4096
lora_rank = 16
lora_alpha = 32

num_gpus = torch.cuda.device_count()
max_memory = {i: "80GiB" for i in range(num_gpus)} if num_gpus else None

offload_dir = Path("offload")
offload_dir.mkdir(exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gpt-oss-120b",
    max_seq_length=max_seq_length,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
    load_in_4bit=True,
    device_map="auto" if num_gpus else {"": "cpu"},
    max_memory=max_memory,
    trust_remote_code=True,
    offload_folder=str(offload_dir) if not num_gpus else None,
)



### Attach LoRA adapters for efficient fine-tuning

We keep the base model frozen and only train LoRA adapters. Increase `lora_rank` on RunPod if memory allows.



In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules="all-linear",
    lora_alpha=lora_alpha,
    lora_dropout=0.05,
    bias="none",
)
FastLanguageModel.for_training(model)



## BIRD dataset overview

The [BIRD benchmark](https://bird-bench.github.io/) provides 9,428 training examples mapping natural language to SQL over large, multi-table databases. We use the public Hugging Face mirror `xu3kev/BIRD-SQL-data-train` for prompts, evidence, and gold SQL queries.



In [ ]:
from datasets import load_dataset

raw_bird_train = load_dataset("xu3kev/BIRD-SQL-data-train", split="train")
print(raw_bird_train)
print(raw_bird_train[0])



### Download and register the BIRD databases

Execution-based rewards require the original SQLite databases. Download them **before** running training on RunPod:

```bash
mkdir -p /workspace/bird_databases
cd /workspace/bird_databases
wget https://bird-bench.oss-cn-beijing.aliyuncs.com/train.zip
unzip train.zip && rm train.zip
```

Set `BIRD_DATABASE_ROOT` to the folder that contains subdirectories such as `train_databases/video_games/video_games.sqlite`.



In [ ]:
import os
from pathlib import Path

BIRD_DATABASE_ROOT = Path(os.getenv("BIRD_DATABASE_ROOT", "/workspace/bird_databases/train_databases"))
print(f"Using BIRD databases from: {BIRD_DATABASE_ROOT}")



### Prompt template and schema utilities

We craft a chat-style prompt that includes the question, any provided evidence, and the full database schema. The model must answer with a single SQL query formatted inside triple backticks.



In [ ]:
import re

SYSTEM_PROMPT = """You are a senior data engineer who translates natural language requests into valid, efficient SQLite SQL. Return only the final SQL query wrapped in triple backticks. Avoid explanations."""

USER_PROMPT_TEMPLATE = """Question: {question}
Evidence: {evidence}
Database schema:
{schema}

Return a single SQL query that answers the question."""

def extract_table_names(schema: str):
    return re.findall(r"CREATE TABLE\s+([A-Za-z0-9_]+)", schema, flags=re.IGNORECASE)



### Build RL-ready records

We precompute prompts, schema metadata, and maximum prompt length so the GRPO trainer can size contexts correctly. Set `DEBUG_EXAMPLES` to a positive integer if you need a quick smoke test without processing all 9,428 examples.



In [ ]:
from datasets import Dataset
from tqdm.auto import tqdm

tokenizer.padding_side = "right"
DEBUG_EXAMPLES = 0  # change to a small number for quick validation

records = []
max_prompt_length = 0
selected_rows = raw_bird_train.select(range(DEBUG_EXAMPLES)) if DEBUG_EXAMPLES else raw_bird_train

for row in tqdm(selected_rows, desc="Formatting prompts"):
    table_names = extract_table_names(row["schema"])
    conversation = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT_TEMPLATE.format(
            question=row["question"].strip(),
            evidence=row["evidence"].strip(),
            schema=row["schema"].strip(),
        )},
    ]
    prompt_text = tokenizer.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=True,
    )
    prompt_length = len(tokenizer(prompt_text, add_special_tokens=False)["input_ids"])
    max_prompt_length = max(max_prompt_length, prompt_length)

    records.append({
        "prompt": conversation,
        "db_id": row["db_id"],
        "reference_sql": row["SQL"].strip(),
        "table_names": table_names,
        "schema": row["schema"],
        "evidence": row["evidence"],
    })

rl_dataset = Dataset.from_list(records)
print(rl_dataset)
print(f"Maximum prompt length: {max_prompt_length} tokens")



We set completion budgets based on the observed prompt lengths.



In [ ]:
maximum_length = max_prompt_length
max_prompt_tokens = maximum_length + 1
max_completion_tokens = max_seq_length - max_prompt_tokens
if max_completion_tokens <= 0:
    raise ValueError(f"max_seq_length={max_seq_length} is too small for prompts ({max_prompt_tokens} tokens)")
print(f"Prompt tokens: {max_prompt_tokens}, completion budget: {max_completion_tokens}")


## Reward shaping for text-to-SQL

We combine three rewards:

1. **Syntax reward** using `sqlglot` to ensure the output parses as SQLite SQL.
2. **Execution reward** comparing predicted and gold answers on the real database.
3. **Schema grounding reward** encouraging use of tables defined in the schema and penalizing hallucinations.



In [ ]:
import re
import sqlite3
from pathlib import Path
from contextlib import closing
from typing import List, Tuple
import sqlglot
from sqlglot import expressions as exp

class QueryTimeout(Exception):
    pass

def extract_sql_from_completion(completion_text: str) -> str:
    text = completion_text.strip()
    if "```" in text:
        blocks = re.findall(r"```(?:sql)?\s*([\s\S]*?)```", text, flags=re.IGNORECASE)
        if blocks:
            text = blocks[-1].strip()
    return text.strip().rstrip(";")

def sanitize_sql(sql: str) -> str:
    if not sql:
        return ""
    forbidden = {"delete", "update", "insert", "alter", "drop", "create", "truncate"}
    lowered = sql.lower()
    if any(token in lowered for token in forbidden):
        return ""
    return sql

def run_sql(sql: str, db_path: Path, fetch_limit: int = 200) -> List[Tuple]:
    if not sql:
        raise ValueError("SQL string is empty")
    with closing(sqlite3.connect(db_path, timeout=1.0)) as conn:
        conn.row_factory = sqlite3.Row
        steps = 0
        def _progress():
            nonlocal steps
            steps += 1
            if steps > 2000:
                raise QueryTimeout("Exceeded SQLite instruction limit")
            return 0
        conn.set_progress_handler(_progress, 1000)
        cur = conn.cursor()
        cur.execute(sql)
        rows = cur.fetchmany(fetch_limit)
        return [tuple(row) for row in rows]


In [ ]:
def syntax_reward(completions, **kwargs):
    scores, diagnostics = [], []
    for completion in completions:
        text = completion[0]["content"]
        sql = sanitize_sql(extract_sql_from_completion(text))
        if not sql:
            scores.append(-0.5)
            diagnostics.append({"error": "empty_or_forbidden"})
            continue
        try:
            sqlglot.parse_one(sql, read="sqlite")
            scores.append(0.5)
            diagnostics.append({"sql": sql})
        except Exception as exc:
            scores.append(-0.5)
            diagnostics.append({"error": str(exc)})
    return scores, diagnostics



In [ ]:
def execution_reward(completions, **kwargs):
    batch = kwargs.get("batch", {})
    db_ids = batch.get("db_id", [])
    references = batch.get("reference_sql", [])
    scores, diagnostics = [], []
    for completion, db_id, reference in zip(completions, db_ids, references):
        text = completion[0]["content"]
        predicted_sql = sanitize_sql(extract_sql_from_completion(text))
        gold_sql = sanitize_sql(reference)
        db_path = BIRD_DATABASE_ROOT / db_id / f"{db_id}.sqlite"
        if not db_path.exists():
            scores.append(-0.1)
            diagnostics.append({"warning": "missing_db", "db_id": db_id})
            continue
        try:
            pred_rows = run_sql(predicted_sql, db_path)
        except Exception as exc:
            scores.append(-1.0)
            diagnostics.append({"error": str(exc), "db_id": db_id})
            continue
        try:
            gold_rows = run_sql(gold_sql, db_path)
        except Exception as exc:
            scores.append(0.0)
            diagnostics.append({"gold_error": str(exc), "db_id": db_id})
            continue
        match = pred_rows == gold_rows
        scores.append(1.0 if match else -0.2)
        diagnostics.append({
            "db_id": db_id,
            "rows_match": match,
            "pred_sample": pred_rows[:3],
            "gold_sample": gold_rows[:3],
        })
    return scores, diagnostics



In [ ]:
def schema_reward(completions, **kwargs):
    batch = kwargs.get("batch", {})
    table_metadata = batch.get("table_names", [])
    scores, diagnostics = [], []
    for completion, allowed_tables in zip(completions, table_metadata):
        text = completion[0]["content"]
        sql = sanitize_sql(extract_sql_from_completion(text))
        if not sql:
            scores.append(-0.1)
            diagnostics.append({"warning": "empty"})
            continue
        try:
            tree = sqlglot.parse_one(sql, read="sqlite")
            used_tables = {t.name.lower() for t in tree.find_all(exp.Table)}
        except Exception:
            scores.append(0.0)
            diagnostics.append({"warning": "parse_failed"})
            continue
        allowed = {t.lower() for t in allowed_tables}
        overlap = len(used_tables & allowed)
        extras = len(used_tables - allowed)
        score = 0.2 * overlap - 0.2 * extras
        scores.append(score)
        diagnostics.append({
            "used_tables": list(used_tables),
            "extras": list(used_tables - allowed),
        })
    return scores, diagnostics



## Configure and instantiate GRPO

We keep the global batch size small because the model is huge, and rely on gradient accumulation plus multiple GPUs to increase effective throughput. Enable bf16 on RunPod for best performance.



In [ ]:
import torch
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    learning_rate=5e-6,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=2,
    max_prompt_length=max_prompt_tokens,
    max_completion_length=max_completion_tokens,
    max_steps=200,
    save_steps=200,
    report_to="none",
    bf16=torch.cuda.is_available(),
    output_dir="outputs",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[syntax_reward, schema_reward, execution_reward],
    args=training_args,
    train_dataset=rl_dataset,
)



### Launch training on RunPod

Run this cell **only** on a GPU-equipped RunPod pod. It will execute GRPO with the rewards defined above.



In [ ]:
# trainer.train()



## Inference sanity check

After training, switch the model to inference mode and generate SQL for a held-out prompt.



In [ ]:
FastLanguageModel.for_inference(model)

demo_example = rl_dataset[0]
input_text = tokenizer.apply_chat_template(
    demo_example["prompt"],
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer([input_text], return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.0, do_sample=False)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))



## Save or push the fine-tuned model

Merge LoRA adapters if you intend to serve with vLLM, or push adapters to the Hub for modular reuse.



In [ ]:
# Example: merge and save LoRA weights
# model.save_pretrained_merged("finetuned_gpt_oss_120b_sql", tokenizer, save_method="lora")
# tokenizer.save_pretrained("finetuned_gpt_oss_120b_sql")

